In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, f1_score
from imblearn.pipeline import Pipeline as ImbPipeline  # Use imblearn's Pipeline
import matplotlib.pyplot as plt
import warnings
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib
import itertools

# Import oversampling techniques from imbalanced-learn
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Load the dataset
# Replace the file path with your actual path
data = pd.read_excel("class1_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Define the feature indexes to be used as predictors
# Note: Pandas uses 0-based indexing
feature_indexes = [24, 20, 16, 15, 12, 3, 10, 13, 21, 9, 22, 28, 29, 19, 38, 2, 6, 26, 31, 5, 33, 11, 35, 34, 1, 8, 36, 32, 30, 23, 14, 0, 4, 27, 37, 18, 17]

# Select the specified features using .iloc
X_selected = X.iloc[:, feature_indexes]

# Create a pipeline with the sampler and classifier
pipeline = ImbPipeline([
    ('sampler', RandomOverSampler()),  # Placeholder, will be set by GridSearchCV
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define the extended parameter grid as a list of dictionaries
param_grid = [
    {
        # Sampler is not 'passthrough', so include sampling_strategy
        'sampler': [SMOTE()],
        'sampler__sampling_strategy': ['auto', 0.5, 0.75, 1.0],
        # Classifier parameters
        'classifier__n_estimators': [150, 200, 250],
        'classifier__criterion': ['gini'],
        'classifier__max_depth': [None] + list(range(5, 36, 10)),  # Depth from 10 to 35 and None
        'classifier__min_samples_split': [3, 5, 7],
        'classifier__min_samples_leaf': [2, 3],
        'classifier__max_features': ['log2'],
        'classifier__bootstrap': [False]
    }
]

# Define the scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'f1': 'f1'
}

# Define the Stratified 10-Fold Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Calculate the number of parameter combinations
# This is the sum of the product of parameters in each dict in param_grid
n_param_combinations = 0
for grid in param_grid:
    n_combinations = 1
    for param in grid:
        n_combinations *= len(grid[param])
    n_param_combinations += n_combinations

# Initialize Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit='roc_auc',  # Use 'roc_auc' to select the best model
    cv=cv,
    n_jobs=-1,  # Use all available cores
    verbose=0,  # Disable sklearn's verbose
    return_train_score=False
)

# Fit the Grid Search to the data without progress bar
print("Starting Grid Search...")
grid_search.fit(X_selected, Y)  # No tqdm or tqdm_joblib here
print("Grid Search Completed.")

# Retrieve the best average AUC and its standard deviation
best_auc = grid_search.best_score_
# Retrieve the standard deviation from cv_results_
# Identify the index of the best parameter set
best_index = grid_search.best_index_
best_auc_std = grid_search.cv_results_['std_test_roc_auc'][best_index]

# Retrieve the best hyperparameters
best_params = grid_search.best_params_

# Retrieve the other metrics for the best parameter set
best_accuracy = grid_search.cv_results_['mean_test_accuracy'][best_index]
best_accuracy_std = grid_search.cv_results_['std_test_accuracy'][best_index]

best_precision = grid_search.cv_results_['mean_test_precision'][best_index]
best_precision_std = grid_search.cv_results_['std_test_precision'][best_index]

best_f1 = grid_search.cv_results_['mean_test_f1'][best_index]
best_f1_std = grid_search.cv_results_['std_test_f1'][best_index]

# Output the results
print(f"\nBest Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")
print(f"Average Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}")
print(f"Average Precision: {best_precision:.4f} ± {best_precision_std:.4f}")
print(f"Average F1 Score: {best_f1:.4f} ± {best_f1_std:.4f}")
print("\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

/home/h_wu4/ml_env/lib/python3.8/site-packages/tqdm_joblib/__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Starting Grid Search...
Grid Search Completed.

Best Average AUC: 0.7809 ± 0.0199
Average Accuracy: 0.8772 ± 0.0087
Average Precision: 0.3193 ± 0.0521
Average F1 Score: 0.3121 ± 0.0529

Best Hyperparameters:
  classifier__bootstrap: False
  classifier__criterion: gini
  classifier__max_depth: 35
  classifier__max_features: log2
  classifier__min_samples_leaf: 2
  classifier__min_samples_split: 3
  classifier__n_estimators: 200
  sampler: SMOTE()
  sampler__sampling_strategy: 0.5


In [2]:
from sklearn.model_selection import cross_val_predict

# Retrieve the best estimator from Grid Search
best_estimator = grid_search.best_estimator_

# Use cross_val_predict to get cross-validated predicted probabilities
# Setting method='predict_proba' and using cv to ensure consistency
print("\nGenerating cross-validated predicted probabilities...")
y_pred_proba = cross_val_predict(best_estimator, X_selected, Y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]

# Define a range of threshold values to evaluate
thresholds = np.linspace(0.0, 1.0, 101)

# Initialize variables to store the best metrics and threshold
best_threshold = 0.5
best_f1_score = 0.0
best_accuracy = 0.0
best_precision = 0.0

print("Optimizing threshold to maximize F1 score...")

for threshold in thresholds:
    # Convert predicted probabilities to binary predictions based on the threshold
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # Calculate F1 score
    current_f1 = f1_score(Y, y_pred)
    
    # Update the best metrics and threshold if current F1 is better
    if current_f1 > best_f1_score:
        best_f1_score = current_f1
        best_threshold = threshold
        best_accuracy = accuracy_score(Y, y_pred)
        best_precision = precision_score(Y, y_pred, zero_division=0)

# Calculate standard deviations using cross-validation
# To compute standard deviations, we'll perform cross-validation predictions and calculate metrics at the best threshold

# Initialize lists to store per-fold metrics
f1_scores = []
accuracies = []
precisions = []

print("\nCalculating metrics at the optimal threshold across folds...")

for fold, (train_idx, test_idx) in enumerate(cv.split(X_selected, Y), 1):
    # Split data
    X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
    y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    
    # Fit the model on the training data
    best_estimator.fit(X_train, y_train)
    
    # Predict probabilities on the test data
    y_proba_fold = best_estimator.predict_proba(X_test)[:, 1]
    
    # Apply the optimal threshold
    y_pred_fold = (y_proba_fold >= best_threshold).astype(int)
    
    # Calculate metrics
    fold_f1 = f1_score(y_test, y_pred_fold)
    fold_accuracy = accuracy_score(y_test, y_pred_fold)
    fold_precision = precision_score(y_test, y_pred_fold, zero_division=0)
    
    # Append to lists
    f1_scores.append(fold_f1)
    accuracies.append(fold_accuracy)
    precisions.append(fold_precision)
    
    print(f"  Fold {fold}: F1={fold_f1:.4f}, Accuracy={fold_accuracy:.4f}, Precision={fold_precision:.4f}")

# Calculate mean and standard deviation for the metrics
mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

mean_precision = np.mean(precisions)
std_precision = np.std(precisions)

# Output the optimized threshold and corresponding metrics
print(f"\n=== Optimized Threshold ===")
print(f"Threshold for Maximum F1 Score: {best_threshold:.2f}")

print(f"\n=== Metrics at Optimal Threshold ===")
print(f"F1 Score: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
print(f"Precision: {mean_precision:.4f} ± {std_precision:.4f}")


Generating cross-validated predicted probabilities...
Optimizing threshold to maximize F1 score...

Calculating metrics at the optimal threshold across folds...
  Fold 1: F1=0.3586, Accuracy=0.8498, Precision=0.2955
  Fold 2: F1=0.3077, Accuracy=0.8544, Precision=0.2703
  Fold 3: F1=0.2835, Accuracy=0.8528, Precision=0.2535
  Fold 4: F1=0.3259, Accuracy=0.8528, Precision=0.2785
  Fold 5: F1=0.3404, Accuracy=0.8495, Precision=0.2824
  Fold 6: F1=0.2740, Accuracy=0.8285, Precision=0.2222
  Fold 7: F1=0.3857, Accuracy=0.8608, Precision=0.3214
  Fold 8: F1=0.3972, Accuracy=0.8625, Precision=0.3333
  Fold 9: F1=0.3206, Accuracy=0.8560, Precision=0.2838
  Fold 10: F1=0.3415, Accuracy=0.8689, Precision=0.3182

=== Optimized Threshold ===
Threshold for Maximum F1 Score: 0.38

=== Metrics at Optimal Threshold ===
F1 Score: 0.3335 ± 0.0380
Accuracy: 0.8536 ± 0.0102
Precision: 0.2859 ± 0.0317


In [3]:
# Open a file in write mode
with open('results_output.txt', 'w') as f:
    # Write the formatted output into the file
    f.write(f"\nBest Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}\n")
    f.write(f"Average Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}\n")
    f.write(f"Average Precision: {best_precision:.4f} ± {best_precision_std:.4f}\n")
    f.write(f"Average F1 Score: {best_f1:.4f} ± {best_f1_std:.4f}\n")
    
    # Write the best hyperparameters
    f.write("\nBest Hyperparameters:\n")
    for param, value in best_params.items():
        f.write(f"  {param}: {value}\n")
    
    # Write the sampling method and sampling rate if applicable
    if 'sampler' in best_params:
        sampler_method = best_params['sampler']
        f.write(f"\n=== Sampling Information ===\n")
        f.write(f"Sampling Method: {sampler_method.__class__.__name__}\n")
        
        if 'sampler__sampling_strategy' in best_params:
            sampling_rate = best_params['sampler__sampling_strategy']
            f.write(f"Sampling Rate: {sampling_rate}\n")
        else:
            f.write(f"Sampling Rate: Not Applicable (No Oversampling)\n")
    else:
        f.write(f"\n=== Sampling Information ===\n")
        f.write(f"Sampling Method: Passthrough (No Oversampling)\n")
    
    # Write optimized threshold
    f.write(f"\n=== Optimized Threshold ===\n")
    f.write(f"Threshold for Maximum F1 Score: {best_threshold:.2f}\n")
    
    # Write the metrics at the optimal threshold
    f.write(f"\n=== Metrics at Optimal Threshold ===\n")
    f.write(f"F1 Score: {mean_f1:.4f} ± {std_f1:.4f}\n")
    f.write(f"Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}\n")
    f.write(f"Precision: {mean_precision:.4f} ± {std_precision:.4f}\n")

In [4]:
# Shut down the virtual machine (Linux-based command)
# import os
# print("Shutting down the VM...")
# os.system(f'echo {password} | sudo -S shutdown -h now')